## 1. Project Vision and Business Question

**Project vision:**  
Help Amazon Appliances product managers identify product listings that may need information review by comparing customer reviews with product metadata.

**Business question:**  
Which appliance listings should be prioritized for product-information review because customers repeatedly mention attributes, problems, or requirements that are missing, unclear, or inconsistent in the product listing?

**Main comparison:**  
Customer review text and ratings compared with product metadata such as product features, descriptions, and details.

**Status:** Done

## 2. Define One Row

**Reviews dataset:**  
One row represents one customer review for one Amazon appliance product.

**Expected identifier:**  
A review can be identified using a combination of `user_id`, `parent_asin`, and `timestamp`.

**Metadata dataset:**  
One row represents one Amazon appliance product listing.

**Expected identifier:**  
`parent_asin` should identify the product listing used to connect metadata with customer reviews.

**Status:** Done

## 3. Preserve the Source

**Data source:**  
McAuley Lab, Amazon Reviews 2023, Appliances category.

**Original source:**  
Hugging Face dataset: McAuley-Lab/Amazon-Reviews-2023.

**Files used:**  
- `Appliances.jsonl` for customer reviews
- `meta_Appliances.jsonl` for product metadata

**Source preservation:**  
The original downloaded JSONL files are kept unchanged. Cleaning and transformations are performed in the notebook on copied DataFrames rather than modifying the source files directly.

**Status:** Done

In [1]:
import pandas as pd

review_reader = pd.read_json(
    "Appliances.jsonl",
    lines=True,
    chunksize=100000
)

reviews = next(review_reader)

reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   rating             100000 non-null  int64         
 1   title              100000 non-null  str           
 2   text               100000 non-null  str           
 3   images             100000 non-null  object        
 4   asin               100000 non-null  str           
 5   parent_asin        100000 non-null  str           
 6   user_id            100000 non-null  str           
 7   timestamp          100000 non-null  datetime64[ms]
 8   helpful_vote       100000 non-null  int64         
 9   verified_purchase  100000 non-null  bool          
dtypes: bool(1), datetime64[ms](1), int64(2), object(1), str(5)
memory usage: 7.0+ MB


In [2]:
meta_reader = pd.read_json(
    "meta_Appliances.jsonl",
    lines=True,
    chunksize=100000
)

metadata = next(meta_reader)

metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 94327 entries, 0 to 94326
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    89651 non-null  str    
 1   title            94327 non-null  str    
 2   average_rating   94327 non-null  float64
 3   rating_number    94327 non-null  int64  
 4   features         94327 non-null  object 
 5   description      94327 non-null  object 
 6   price            46726 non-null  float64
 7   images           94327 non-null  object 
 8   videos           94327 non-null  object 
 9   store            93411 non-null  str    
 10  categories       94327 non-null  object 
 11  details          94327 non-null  object 
 12  parent_asin      94327 non-null  str    
 13  bought_together  0 non-null      float64
 14  subtitle         5 non-null      str    
 15  author           1 non-null      object 
dtypes: float64(3), int64(1), object(7), str(5)
memory usage: 11.5+ MB


In [3]:
metadata.isna().sum()

main_category       4676
title                  0
average_rating         0
rating_number          0
features               0
description            0
price              47601
images                 0
videos                 0
store                916
categories             0
details                0
parent_asin            0
bought_together    94327
subtitle           94322
author             94326
dtype: int64

In [4]:
metadata[["bought_together", "subtitle", "author"]].head(20)

,bought_together,subtitle,author
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
5,NaN,NaN,NaN
6,NaN,NaN,NaN
7,NaN,NaN,NaN
8,NaN,NaN,NaN
9,NaN,NaN,NaN


### Cleaning Rule: Sparse Metadata Columns

- Drop `bought_together` because it is 100% missing.
- Drop `subtitle` because only 5 of 94,327 records contain a value.
- Drop `author` because only 1 of 94,327 records contains a value.

**Reason:**  
These columns are almost entirely missing and do not provide reliable information for the project analysis.

**Check:**  
Verify that the three columns are removed and that the row count remains unchanged.

In [5]:
clean_metadata = metadata.drop(
    columns=["bought_together", "subtitle", "author"]
)

clean_metadata.shape

(94327, 13)

In [6]:
clean_metadata.isna().sum()

main_category      4676
title                 0
average_rating        0
rating_number         0
features              0
description           0
price             47601
images                0
videos                0
store               916
categories            0
details               0
parent_asin           0
dtype: int64

In [7]:
(clean_metadata.isna().sum() / len(clean_metadata) * 100).round(2)

main_category      4.96
title              0.00
average_rating     0.00
rating_number      0.00
features           0.00
description        0.00
price             50.46
images             0.00
videos             0.00
store              0.97
categories         0.00
details            0.00
parent_asin        0.00
dtype: float64

In [8]:
clean_metadata["main_category"].value_counts(dropna=False).head(20)

main_category
Tools & Home Improvement     42694
Appliances                   25572
Amazon Home                  13915
Industrial & Scientific       5521
NaN                           4676
Automotive                     568
Health & Personal Care         268
All Electronics                168
Sports & Outdoors              124
Grocery                         95
Cell Phones & Accessories       85
Musical Instruments             83
Baby                            82
AMAZON FASHION                  71
All Beauty                      68
Office Products                 68
Computers                       57
Pet Supplies                    49
Camera & Photo                  39
Toys & Games                    38
Name: count, dtype: int64

In [9]:
clean_metadata["price"].describe()

count    46726.000000
mean        86.479954
std        325.318397
min          0.010000
25%         14.990000
50%         26.990000
75%         58.130000
max      21095.620000
Name: price, dtype: float64

In [10]:
clean_metadata["price"].sort_values(ascending=False).head(20)

92138    21095.62
45697    10960.61
81012     8458.86
42312     7880.48
47869     7787.75
15959     7691.01
23498     7373.77
7481      7278.00
13687     6999.01
17248     6999.00
12514     6399.00
44550     6399.00
35793     6349.00
27587     6336.00
53489     6099.99
77725     5868.16
34434     5799.00
91043     5789.00
42524     5760.00
14221     5759.00
Name: price, dtype: float64

In [11]:
clean_metadata[
    ["title", "main_category", "store", "price", "parent_asin"]
].sort_values(
    "price",
    ascending=False
).head(20)

,title,main_category,store,price,parent_asin
92138,TurboChef BULLET Rapid Cook Electric Microwave...,Appliances,TurboChef,21095.62,B01D05UB8M
45697,Turbo Air TCGB-48-DR - 48-Inch Curved Glass Dr...,Appliances,Turbo Air,10960.61,B008EMB14K
81012,"Multi Refrigerant (HFO1234yf, R134a, R22, and ...",Tools & Home Improvement,Artist Unknown,8458.86,B00BHCFM94
42312,Turbo Air M3F72-3-N commercial kitchen upright...,Industrial & Scientific,Turbo Air,7880.48,B0C62F9ZNH
47869,Scotsman N0622A-1 Prodigy Plus Nugget-Style Ic...,Industrial & Scientific,Scotsman,7787.75,B00EPNQOIK
15959,"40"" Majestic II Series Freestanding Dual Fuel ...",Appliances,Ilve,7691.01,B081VXNFLB
23498,Hoshizaki F2A-FS 55-Inch Two Door Solid Doors ...,Industrial & Scientific,Hoshizaki,7373.77,B07QHMQ3M9
7481,Thor Kitchen HRG4808UHRH4806U 2-Piece Kitchen ...,Appliances,Thor Kitchen,7278.00,B06Y411RD3
13687,"36"" Majestic II Series Freestanding Dual Fuel ...",Appliances,Ilve,6999.01,B082BDJ7MB
17248,"UPW120FDMPI 48"" Professional Plus Dual Fuel Ra...",Appliances,Ilve,6999.00,B081VSQDJM


### Cleaning Decision: Price

- `price` is missing for 47,601 of 94,327 records (50.46%).
- Missing prices are kept as missing rather than filled because more than half of the values are unavailable and imputing a mean or median would create artificial prices.
- Very high prices were reviewed manually. The highest-priced records correspond to commercial or professional appliances such as commercial microwaves, refrigerators, ice machines, and ranges.
- These high prices are therefore retained rather than treated as errors.

**Check:**  
Reviewed the top 20 highest-price records and confirmed that the values are plausible for the listed products.

### Cleaning Decision: Missing Category and Store Values

- `main_category` is missing for 4,676 records (4.96%).
- `store` is missing for 916 records (0.97%).
- These missing values are kept as missing because there is no reliable rule for inferring the correct category or store.
- Missing `main_category` values are not automatically filled as `Appliances` because the dataset contains multiple main-category labels, including Tools & Home Improvement, Amazon Home, and Industrial & Scientific.

**Check:**  
Reviewed the distribution of `main_category` values and confirmed that multiple category labels are present.

In [12]:
clean_metadata["parent_asin"].duplicated().sum()

np.int64(0)

In [13]:
metadata_check = clean_metadata[
    ["title", "average_rating", "rating_number", "price",
     "store", "parent_asin"]
]

metadata_check.duplicated().sum()

np.int64(0)

In [14]:
reviews.duplicated(
    subset=["user_id", "parent_asin", "timestamp"]
).sum()

np.int64(323)

In [15]:
duplicate_reviews = reviews[
    reviews.duplicated(
        subset=["user_id", "parent_asin", "timestamp"],
        keep=False
    )
].sort_values(
    ["user_id", "parent_asin", "timestamp"]
)

duplicate_reviews.head(20)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
49528,5,Five Stars,"Fit perfect and works great, really fast delivery",[],B0089VAAF2,B0089VAAF2,AE2B7AQSZED3AYMTREETFR4JEVRQ,2015-04-28 11:36:13.000,0,True
49534,5,Five Stars,"Fit perfect and works great, really fast delivery",[],B0089VAAF2,B0089VAAF2,AE2B7AQSZED3AYMTREETFR4JEVRQ,2015-04-28 11:36:13.000,0,True
49527,5,Five Stars,nice fit for our Samsung refrig,[],B0089VAAIY,B0089VAAIY,AE2B7AQSZED3AYMTREETFR4JEVRQ,2016-12-06 21:54:01.000,0,True
49533,5,Five Stars,nice fit for our Samsung refrig,[],B0089VAAIY,B0089VAAIY,AE2B7AQSZED3AYMTREETFR4JEVRQ,2016-12-06 21:54:01.000,0,True
68904,5,Great Coffee with these filer circles,"Perfect match for Aero Press, small, saving pa...",[],B000LTOCSG,B0B57GSDMD,AE2PAJKBBL3H3APRJR7XC2JHOAOQ,2020-04-20 11:24:59.683,0,True
68905,5,Great Coffee with these filer circles,"Perfect match for Aero Press, small, saving pa...",[],B000LTOCSG,B0B57GSDMD,AE2PAJKBBL3H3APRJR7XC2JHOAOQ,2020-04-20 11:24:59.683,0,True
65194,5,Tiny size. Easy setup,Paired this into a HomeAssistant setup. Quick ...,[],B0987MNJZ1,B0987MNJZ1,AE346DRDWBR4GQ7WELOVEJCWFPBQ,2023-02-19 22:51:23.743,0,True
65212,5,Tiny size. Easy setup,Paired this into a HomeAssistant setup. Quick ...,[],B0987MNJZ1,B0987MNJZ1,AE346DRDWBR4GQ7WELOVEJCWFPBQ,2023-02-19 22:51:23.743,0,True
57499,5,I like this pitcher shape,This pitcher is shaped differently than it is ...,[],B01N1ZKH2W,B07PFFBN8P,AE374LE7XANMIUY5B4FENPQRFGLA,2021-10-31 13:42:27.723,0,True
57500,5,I like this pitcher shape,This pitcher is shaped differently than it is ...,[],B01N1ZKH2W,B07PFFBN8P,AE374LE7XANMIUY5B4FENPQRFGLA,2021-10-31 13:42:27.723,0,True


In [16]:
reviews_check = reviews.copy()
reviews_check["images"] = reviews_check["images"].astype(str)

reviews_check.duplicated().sum()

np.int64(323)

### Cleaning Rule: Duplicate Reviews

- In the first 100,000 review records, 323 rows have repeated `user_id + parent_asin + timestamp`.
- A full-row duplicate check also identified exactly 323 duplicate rows.
- Because the repeated records are exact duplicates across all fields, duplicate copies will be removed and one copy will be kept.

**Reason:**  
Exact duplicate reviews would otherwise double-count the same customer feedback and could bias review counts, rating summaries, and text analysis.

**Check:**  
Compare row counts before and after removing duplicates and confirm that no exact duplicate rows remain.

In [18]:
mask = reviews_check.duplicated()

clean_reviews = reviews.loc[~mask].copy()

clean_reviews.shape

(99677, 10)

In [19]:
clean_reviews_check = clean_reviews.copy()
clean_reviews_check["images"] = clean_reviews_check["images"].astype(str)

clean_reviews_check.duplicated().sum()

np.int64(0)

In [20]:
clean_reviews.isna().sum()

rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
dtype: int64

In [21]:
clean_reviews["rating"].value_counts().sort_index()

rating
1     7911
2     3304
3     5340
4    10997
5    72125
Name: count, dtype: int64

In [22]:
clean_reviews["helpful_vote"].describe()

count    99677.000000
mean         1.082466
std          9.506068
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max       1138.000000
Name: helpful_vote, dtype: float64

In [23]:
clean_reviews[
    ["title", "text", "rating", "helpful_vote", "parent_asin"]
].sort_values(
    "helpful_vote",
    ascending=False
).head(10)

,title,text,rating,helpful_vote,parent_asin
7722,Great for vegans too!,I'm vegan so I don't buy eggs. Nevertheless as...,5,1138,B00GN92KQ4
78730,Broke in Less Than a Month!! :(,Purchased for my Mom for Christmas and it has ...,1,1067,B0751KDVTP
23735,Love it!,I needed one of these ice makers because I hav...,4,678,B07MWTNDXT
62936,Buyer Beware,"BUYER BEWARE!!!!!!!!!!!!!<br />On 06/29/2016, ...",1,615,B08HQBR75H
29801,1qt of ice takes 90 minutes,I wanted to post an actual timetable for what ...,4,602,B09QYZ18YD
11414,"Thank you, Amazon, for Prime Day, which enable...","The freezer arrived in perfect condition, desp...",5,593,B0B4YS19SH
24648,Little bit bigger than I thought but it’s AMAZING,I love love love this washer!! We have a pipe ...,5,515,B07B94ZR74
87698,Tapping water source from the shower,The washer is pretty quiet and doesn't vibrate...,5,445,B002UYSHMM
33981,"ADVANCED Install / Setup WORKS GREAT (read on,...",**Original Review**<br />We ordered the larger...,5,428,B00KY5V4M4
4311,FINALLY - The Right Stuff,I have been through three of these coffee drip...,5,425,B08JCCD38P


### Cleaning Decision: Helpful Votes

- `helpful_vote` has no negative values.
- Most reviews have 0 helpful votes, but a small number have very high counts.
- The highest values were manually reviewed and correspond to plausible customer reviews rather than obvious data errors.
- High helpful-vote values are therefore retained.

**Check:**  
Reviewed the top 10 records by `helpful_vote`; no clearly invalid values were found.

In [24]:
clean_reviews["timestamp"].min(), clean_reviews["timestamp"].max()

(Timestamp('2003-06-04 04:14:46'), Timestamp('2023-03-20 21:51:42.143000'))

### Cleaning Decision: Review Timestamps

- Review timestamps range from June 4, 2003 to March 20, 2023.
- No future dates or clearly invalid dates were found.
- All timestamps are retained without modification.

**Check:**  
Compared the minimum and maximum review timestamps and confirmed that the observed range is plausible.

In [25]:
title_spaces = (clean_reviews["title"] != clean_reviews["title"].str.strip()).sum()
text_spaces = (clean_reviews["text"] != clean_reviews["text"].str.strip()).sum()

title_spaces, text_spaces

(np.int64(1), np.int64(21))

In [26]:
clean_reviews.loc[
    clean_reviews["text"] != clean_reviews["text"].str.strip(),
    ["title", "text"]
].head(10)

,title,text
2429,They came with one broken,
24702,Arrived damaged. Seller has not responded to m...,
27002,Good quality and bright color,
42026,"Kinda loud, kinda need a lint roller, but over...",
47456,Great suction and lights,
47648,Cheap paper filters,
58827,Quick ice,
62727,an ultra quiet product,
64378,"Is it normal when I soak it for 15 mins, the w...",
65434,Mother’s Day Gift!,


In [27]:
clean_reviews["title"] = clean_reviews["title"].str.strip()
clean_reviews["text"] = clean_reviews["text"].str.strip()

In [28]:
title_spaces_after = (
    clean_reviews["title"] != clean_reviews["title"].str.strip()
).sum()

text_spaces_after = (
    clean_reviews["text"] != clean_reviews["text"].str.strip()
).sum()

title_spaces_after, text_spaces_after

(np.int64(0), np.int64(0))

### Cleaning Decision: Standardize Review Text

- Before cleaning, 1 `title` value and 21 `text` values contained leading or trailing whitespace.
- Leading and trailing whitespace was removed using `.str.strip()`.
- Internal wording, capitalization, punctuation, and meaningful differences were preserved.

**Check:**  
After cleaning, both `title` and `text` had 0 remaining values with leading or trailing whitespace.

In [29]:
merged = clean_reviews.merge(
    clean_metadata,
    on="parent_asin",
    how="left",
    suffixes=("_review", "_meta")
)

merged.shape

(99677, 22)

In [30]:
merged["title_meta"].isna().sum()

np.int64(0)

### Merge Verification

- Reviews and metadata were merged using `parent_asin`.
- The expected relationship is many reviews to one metadata record.
- `parent_asin` was unique in the metadata table.
- Clean review row count before merge: 99,677.
- Row count after the left merge: 99,677.
- Unmatched review records: 0.
- No unexpected extra rows were created by the merge.

**Status:** Done

In [31]:
starting_reviews = len(reviews)
final_reviews = len(clean_reviews)

starting_metadata = len(metadata)
final_metadata = len(clean_metadata)

starting_reviews, final_reviews, starting_metadata, final_metadata

(100000, 99677, 94327, 94327)

In [32]:
total_review_rows = 0

for chunk in pd.read_json(
    "Appliances.jsonl",
    lines=True,
    chunksize=100000
):
    total_review_rows += len(chunk)

total_review_rows

2128605

In [33]:
seen_hashes = set()
full_duplicate_count = 0

for chunk in pd.read_json(
    "Appliances.jsonl",
    lines=True,
    chunksize=100000
):
    check_chunk = chunk.copy()
    check_chunk["images"] = check_chunk["images"].astype(str)

    row_hashes = pd.util.hash_pandas_object(
        check_chunk,
        index=False
    )

    for h in row_hashes:
        if h in seen_hashes:
            full_duplicate_count += 1
        else:
            seen_hashes.add(h)

full_duplicate_count

22656

### Cleaning Rule: Duplicate Reviews

- The full Appliances review dataset contains 2,128,605 rows.
- 22,656 exact duplicate rows were identified across the full dataset.
- Exact duplicate copies will be removed while one copy of each review is retained.
- The expected cleaned review count after duplicate removal is 2,105,949 rows.

**Reason:**  
Exact duplicate reviews would double-count the same customer feedback and could bias review counts, rating summaries, and text analysis.

**Check:**  
Verify that the cleaned full review dataset contains 2,105,949 rows and that no exact duplicate rows remain.

In [35]:
seen_hashes = set()
clean_chunks = []

for chunk in pd.read_json(
    "Appliances.jsonl",
    lines=True,
    chunksize=100000
):
    chunk = chunk.copy()

    # standardize text
    chunk["title"] = chunk["title"].str.strip()
    chunk["text"] = chunk["text"].str.strip()

    # make rows hashable for duplicate checking
    check_chunk = chunk.copy()
    check_chunk["images"] = check_chunk["images"].astype(str)

    row_hashes = pd.util.hash_pandas_object(
        check_chunk,
        index=False
    )

    keep_mask = []

    for h in row_hashes:
        if h in seen_hashes:
            keep_mask.append(False)
        else:
            seen_hashes.add(h)
            keep_mask.append(True)

    clean_chunks.append(
        chunk.loc[keep_mask]
    )

full_clean_reviews = pd.concat(
    clean_chunks,
    ignore_index=True
)

full_clean_reviews.shape

(2105949, 10)

In [36]:
seen_hashes = set()
clean_chunks = []

for chunk in pd.read_json(
    "Appliances.jsonl",
    lines=True,
    chunksize=100000
):
    chunk = chunk.copy()

    # Standardize text
    chunk["title"] = chunk["title"].str.strip()
    chunk["text"] = chunk["text"].str.strip()

    # Create hashable version for duplicate detection
    check_chunk = chunk.copy()
    check_chunk["images"] = check_chunk["images"].astype(str)

    row_hashes = pd.util.hash_pandas_object(
        check_chunk,
        index=False
    )

    keep_mask = []

    for h in row_hashes:
        if h in seen_hashes:
            keep_mask.append(False)
        else:
            seen_hashes.add(h)
            keep_mask.append(True)

    clean_chunks.append(
        chunk.loc[keep_mask]
    )

full_clean_reviews = pd.concat(
    clean_chunks,
    ignore_index=True
)

full_clean_reviews.shape

(2105949, 10)

### Reconcile Changes

**Reviews**
- Starting row count: 2,128,605
- Exact duplicate rows removed: 22,656
- Final cleaned row count: 2,105,949

**Metadata**
- Starting row count: 94,327
- Final row count: 94,327
- No metadata rows were removed.
- Three highly sparse columns (`bought_together`, `subtitle`, and `author`) were removed, reducing the metadata from 16 columns to 13 columns.

**Explanation of changes**
- Review row count decreased only because exact duplicate reviews were removed.
- Leading and trailing whitespace in review `title` and `text` fields was standardized without changing row count.
- Metadata row count remained unchanged because missing values and high-price records were retained when there was no reliable basis for deletion or imputation.

**Status:** Done

In [40]:
calculation_checks = {
    "review_count": len(full_clean_reviews),
    "rating_count": full_clean_reviews["rating"].count(),
    "average_rating": full_clean_reviews["rating"].mean(),

    "metadata_count": len(clean_metadata),
    "price_non_missing_count": clean_metadata["price"].count(),
    "average_price_non_missing_only": clean_metadata["price"].mean(),

    "main_category_non_missing_count": clean_metadata["main_category"].count(),
    "store_non_missing_count": clean_metadata["store"].count()
}

calculation_checks

{'review_count': 2105949,
 'rating_count': np.int64(2105949),
 'average_rating': np.float64(4.222639769529081),
 'metadata_count': 94327,
 'price_non_missing_count': np.int64(46726),
 'average_price_non_missing_only': np.float64(86.47995398707359),
 'main_category_non_missing_count': np.int64(89651),
 'store_non_missing_count': np.int64(93411)}

### 11. Check Calculations

- The cleaned review dataset contains 2,105,949 records.
- All 2,105,949 reviews have a valid rating, so the average review rating uses the full cleaned review dataset.
- The average review rating is approximately 4.22.

- The metadata dataset contains 94,327 records.
- Only 46,726 records have a non-missing `price`, so the average price of approximately $86.48 is calculated only from those 46,726 records.
- Missing prices are excluded from the denominator rather than being filled with artificial values.
- `main_category` is available for 89,651 metadata records, and `store` is available for 93,411 records.
- Analyses grouped by `main_category` or `store` should therefore exclude missing group keys unless the analysis explicitly treats missing values as a separate group.

**Status:** Done

In [41]:
record_checks = pd.DataFrame({
    "check": [
        "duplicate_review_removed",
        "review_text_trimmed",
        "high_price_retained",
        "missing_price_retained",
        "merge_match_success"
    ],
    "expected": [
        "Only one copy should remain",
        "Leading/trailing whitespace should be removed",
        "High but plausible price should be retained",
        "Missing price should remain missing",
        "Review should match metadata by parent_asin"
    ]
})

record_checks

,check,expected
0,duplicate_review_removed,Only one copy should remain
1,review_text_trimmed,Leading/trailing whitespace should be removed
2,high_price_retained,High but plausible price should be retained
3,missing_price_retained,Missing price should remain missing
4,merge_match_success,Review should match metadata by parent_asin


In [44]:
# Record Check 1: Exact duplicate review
dup_example = duplicate_reviews.iloc[0]

dup_key = (
    (full_clean_reviews["user_id"] == dup_example["user_id"]) &
    (full_clean_reviews["parent_asin"] == dup_example["parent_asin"]) &
    (full_clean_reviews["timestamp"] == dup_example["timestamp"])
)

duplicate_actual = full_clean_reviews.loc[dup_key].shape[0]


# Record Check 2: High but plausible price retained
high_price_row = metadata.loc[metadata["price"].idxmax()]

high_price_original = high_price_row["price"]

high_price_actual = clean_metadata.loc[
    clean_metadata["parent_asin"] == high_price_row["parent_asin"],
    "price"
].iloc[0]


# Record Check 3: Missing price retained
missing_price_row = metadata.loc[
    metadata["price"].isna()
].iloc[0]

missing_price_asin = missing_price_row["parent_asin"]

missing_price_actual = clean_metadata.loc[
    clean_metadata["parent_asin"] == missing_price_asin,
    "price"
].iloc[0]


# Record Check 4: Missing main_category retained
missing_category_row = metadata.loc[
    metadata["main_category"].isna()
].iloc[0]

missing_category_asin = missing_category_row["parent_asin"]

missing_category_actual = clean_metadata.loc[
    clean_metadata["parent_asin"] == missing_category_asin,
    "main_category"
].iloc[0]


# Record Check 5: Review successfully matches metadata
merge_asin = full_clean_reviews["parent_asin"].iloc[0]

merge_actual = clean_metadata.loc[
    clean_metadata["parent_asin"] == merge_asin
].shape[0]


# Create final record-check table
record_checks = pd.DataFrame({
    "check": [
        "duplicate_review_removed",
        "high_price_retained",
        "missing_price_retained",
        "missing_category_retained",
        "merge_match_success"
    ],

    "original": [
        "Duplicate review appeared more than once",
        high_price_original,
        "NaN",
        "NaN",
        merge_asin
    ],

    "expected": [
        "Only one copy should remain",
        "High but plausible price should remain unchanged",
        "Missing price should remain NaN",
        "Missing main_category should remain NaN",
        "Exactly 1 metadata record should match"
    ],

    "actual": [
        f"{duplicate_actual} copy remains",
        high_price_actual,
        missing_price_actual,
        missing_category_actual,
        f"{merge_actual} metadata match"
    ],

    "match": [
        duplicate_actual == 1,
        high_price_actual == high_price_original,
        pd.isna(missing_price_actual),
        pd.isna(missing_category_actual),
        merge_actual == 1
    ]
})

record_checks

,check,original,expected,actual,match
0,duplicate_review_removed,Duplicate review appeared more than once,Only one copy should remain,1 copy remains,True
1,high_price_retained,21095.62,High but plausible price should remain unchanged,21095.62,True
2,missing_price_retained,NaN,Missing price should remain NaN,NaN,True
3,missing_category_retained,NaN,Missing main_category should remain NaN,NaN,True
4,merge_match_success,B01N0TQ0OH,Exactly 1 metadata record should match,1 metadata match,True


In [45]:
record_checks[["check", "match"]]

,check,match
0,duplicate_review_removed,True
1,high_price_retained,True
2,missing_price_retained,True
3,missing_category_retained,True
4,merge_match_success,True


### 12. Verify Individual Records

Five individual records were checked, including difficult cases:

1. An exact duplicate review was expected to retain only one copy after cleaning.
2. A very high but plausible appliance price was expected to remain unchanged.
3. A missing price was expected to remain missing rather than being imputed.
4. A missing `main_category` value was expected to remain missing because no reliable category could be inferred.
5. A review `parent_asin` was expected to match exactly one metadata record.

All five actual results matched the expected results shown in the record-check table.

**Status:** Done